# Relevancy Pipeline NLP - News Benchmark & Evaluation
Project Capstone: Rifki Afina Putri, S.T., M.S., Ph.D.
* **Input File Tunggal**: `news baru - News fixx.csv`.
* **Model**: Google AI Studio API (`gemma-4-26b-a4b-it` / Gemini API).
* **Ground Truth Rule**: `Stance1` in [`SUPPORT`, `OPPOSE`, `NEUTRAL`] -> **Relevant** (`True`). `Irrelevant` -> **Irrelevant** (`False`).
* **Arsitektur Murni LLM (Web-Deployable)**: Menggunakan ekstraksi konteks deterministik KWIC (Key Word In Context) tanpa dependensi lokal yang berat (tanpa BERT).
* **Stateful & Resumable Checkpoint**: Menyimpan progres langsung ke `relevancy_checkpoint.csv`. Jika batas kuota API tercapai, program berhenti aman dan dapat dilanjutkan tanpa memproses ulang yang sudah selesai.

## 1. Import Libraries & Muat Dataset Input
Memuat dataset langsung dari file input tunggal: `news baru - News fixx.csv`.

In [1]:
import pandas as pd
import numpy as np
import os
import re

# =============================================================================
# FILE INPUT TUNGGAL
# =============================================================================
INPUT_FILE = 'news baru - News fixx.csv'

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"File '{INPUT_FILE}' tidak ditemukan di direktori kerja saat ini.")

# Baca file dengan deteksi encoding otomatis
df_matched = pd.DataFrame()
for enc in ['utf-8', 'latin1', 'cp1252']:
    for sep in [',', ';', None]:
        try:
            df_try = pd.read_csv(INPUT_FILE, sep=sep, engine='python' if sep is None else 'c', encoding=enc)
            if df_try.shape[1] > 1 and len(df_try) > 0:
                df_matched = df_try
                break
        except Exception:
            continue
    if len(df_matched) > 0:
        break

df_matched.columns = df_matched.columns.astype(str).str.strip()
df_annot = df_matched.copy()  # df_annot dan df_matched bersumber dari dataset tunggal ini

print(f"[Sukses] Berhasil memuat dataset dari '{INPUT_FILE}'")
print(f"Total baris: {len(df_matched)} | Total kolom: {df_matched.shape[1]}")
print(f"Daftar kolom: {df_matched.columns.tolist()[:10]}")
display(df_matched.head(3))


[Sukses] Berhasil memuat dataset dari 'news baru - News fixx.csv'
Total baris: 42 | Total kolom: 24
Daftar kolom: ['no', 'source', 'url', 'title', 'date', 'keyword', 'target', 'format', 'content', 'Actor1']


,no,source,url,title,date,keyword,target,format,content,Actor1,...,Evidence2,Actor3,Stance3,Evidence3,Actor4,Stance4,Evidence4,Actor5,Stance5,Evidence5
0,1,detik,https://www.detik.com/bali/bisnis/d-7972473/pl...,"PLTM Santong Hasilkan 3,5 Juta kWh Per Tahun d...",2025-06-19 16:36:00,PLTU,PLTU,article,Pembangkit Listrik Tenaga Mikrohidro (PLTM) Sa...,General Manager (GM) PT Perusahaan Listrik Ne...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,detik,https://news.detik.com/berita/d-6900154/menter...,"Menteri LHK: Setelah Hujan, Kualitas Udara di ...",2023-08-28 16:34:00,PLTU,PLTU,article,"Untuk mengatasi polusi udara di Jakarta, Bogor...",Menteri Lingkungan Hidup dan Kehutanan (LHK) S...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,kompas,https://nasional.kompas.com/read/2020/01/16/10...,YLBHI Sebut Ada 26 Kasus Pencemaran Udara pada...,2020-01-16 10:27:00,PLTU,PLTU,article,"JAKARTA, KOMPAS.com- Yayasan Lembaga Bantuan H...",Ketua Bidang Manajemen PengetahuanYLBHIRahma M...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Validasi & Standarisasi Kolom
Memastikan kolom `title`, `content`, dan `Stance1` tersedia dan bersih untuk dievaluasi.

In [2]:
# Validasi dan standarisasi nama kolom 'title'
def find_column(df, candidates):
    for col in df.columns:
        if col.lower() in candidates:
            return col
    return None

title_col = find_column(df_matched, ['title', 'judul', 'news_title'])
content_col = find_column(df_matched, ['content', 'isi', 'berita', 'article', 'text'])

if title_col and title_col != 'title':
    df_matched['title'] = df_matched[title_col]
    df_annot['title'] = df_annot[title_col]

if content_col and content_col != 'content':
    df_matched['content'] = df_matched[content_col]
    df_annot['content'] = df_annot[content_col]

# Hapus baris dengan judul atau konten kosong
initial_len = len(df_matched)
df_matched = df_matched.dropna(subset=['title']).reset_index(drop=True)
df_annot = df_matched.copy()

print(f"Jumlah artikel siap diproses untuk cek relevansi: {len(df_matched)} / {initial_len}")
display(df_matched[['no', 'title']].head(5))


Jumlah artikel siap diproses untuk cek relevansi: 42 / 42


,no,title
0,1,"PLTM Santong Hasilkan 3,5 Juta kWh Per Tahun d..."
1,2,"Menteri LHK: Setelah Hujan, Kualitas Udara di ..."
2,3,YLBHI Sebut Ada 26 Kasus Pencemaran Udara pada...
3,24,"Cegah Listrik Mati Bergilir, PLTU Dimodifikasi..."
4,4,"Ekspor Batu Bara Dilarang, Ini Kata Adaro"


## 3. Set Up Google GenAI Client & API Key

In [3]:
from google import genai
from google.genai import types
import time
import json
import os

# Ambil API key dari Colab Secrets atau Environment Variables
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get('GEMMA') or userdata.get('GEMINI_API_KEY')
except Exception:
    pass

if not api_key:
    api_key = os.environ.get('GEMINI_API_KEY') or os.environ.get('GEMMA') or 'AQ.Ab8RN6Krmtg1nxmqnm2fa3S9Q8uD7lpJHCTR5jbjg0xIvcmO-g'

client = genai.Client(api_key=api_key)
print('Google GenAI Client siap.')


Google GenAI Client siap.


## 4. Kompresi Konteks Ekstraktif (KWIC) & Schema Model Rubrik Tergeneralisasi
* **Broadened Lexical Filter**: Menggunakan pola akar kata energi yang diperluas untuk data skala besar (1.000+ baris) agar tidak terjadi false dismissal.
* **Rubric-Based Prompt**: Tidak menggunakan aturan overfitting kasus kecil (tanpa menyebut nama kasus spesifik). Berbasis rubrik substantif: kebijakan, emisi, dekarbonisasi, pensiun dini, dan pendanaan transisi energi di Indonesia.
* **Structured Output**: `{id: int, relevancy: bool, confidence: float, evidence_quote: str, reasoning: str}`.

In [4]:
from pydantic import BaseModel, Field
from typing import List
import re
import time

# Exception khusus untuk mendeteksi habisnya kuota akun secara permanen
class QuotaExhaustedError(Exception):
    """Dilempar saat batas kuota API (ResourceExhausted / Daily Limit) tercapai."""
    pass

# Regex pola kata kunci energi Indonesia yang digeneralisasi (mencegah false dismissals pada 1.000+ baris)
KEYWORDS_PATTERN = re.compile(
    r'\b(pltu|pltn|pltgu|uap|nuklir|batu\s*bara|batubara|coal|smr|transisi\s*energi|bauran\s*energi|ebt|dekarbonisasi|emisi\s*karbon|fisi|reaktor|pembangkit)\b',
    re.IGNORECASE
)

def extract_topic_salient_context(title: str, content: str, max_chars: int = 1500) -> str:
    """
    Kompresor konteks deterministik lokal (Key Word In Context / KWIC):
    1. Mempertahankan Judul dan kalimat pembuka (lead paragraph).
    2. Mengambil semua kalimat yang memuat kata kunci energi beserta 1 kalimat sebelum & sesudahnya.
    3. Mengurangi ukuran token hingga ~73% sekaligus menjamin bukti relevan tidak hilang.
    """
    title_str = str(title).strip()
    content_str = str(content).strip()

    sentences = re.split(r'(?<=[.!?])\s+', content_str)
    if not sentences or len(sentences) == 0:
        return title_str

    selected_indices = set()
    selected_indices.add(0)
    if len(sentences) > 1 and len(sentences[0]) < 150:
        selected_indices.add(1)

    for idx, sentence in enumerate(sentences):
        if KEYWORDS_PATTERN.search(sentence):
            if idx > 0:
                selected_indices.add(idx - 1)
            selected_indices.add(idx)
            if idx < len(sentences) - 1:
                selected_indices.add(idx + 1)

    ordered_sentences = [sentences[i] for i in sorted(selected_indices)]
    condensed_body = " [...] ".join(ordered_sentences)
    return condensed_body[:max_chars]

class RelevancyItem(BaseModel):
    id: int = Field(description='The index ID of the analyzed item.')
    relevancy: bool = Field(description='True if the news article discusses PLTN (nuclear) or PLTU (coal) in Indonesia, False otherwise.')
    confidence: float = Field(description='Confidence score between 0.0 and 1.0.', ge=0.0, le=1.0)
    evidence_quote: str = Field(description='Exact verbatim phrase or sentence mentioning PLTU/PLTN/batubara/nuklir in Indonesia. Empty if irrelevant.')
    reasoning: str = Field(description='Brief 1-sentence reasoning explaining why the article is relevant or irrelevant.')

class RelevancyBatchResponse(BaseModel):
    results: List[RelevancyItem] = Field(description='List of evaluation results for the batch.')

GENERALIZED_SYSTEM_PROMPT = (
    "You are an expert NLP assistant specializing in Indonesian energy policy and power infrastructure.\n"
    "Your task is to classify whether each News Article is RELEVANT or IRRELEVANT to:\n"
    "- PLTU (Pembangkit Listrik Tenaga Uap / coal-fired power plants) in Indonesia, OR\n"
    "- PLTN (Pembangkit Listrik Tenaga Nuklir / nuclear power plants) in Indonesia.\n\n"
    "RUBRIC FOR ACCURATE CLASSIFICATION:\n"
    "1. RELEVANT (relevancy = true):\n"
    "   - The article substantively reports on, discusses, or analyzes PLTU or PLTN in Indonesia.\n"
    "   - Includes: operations, capacity, emissions, air pollution, co-firing, coal supply shortages, public debates, "
    "     early retirement (pensiun dini), decarbonization roadmaps, transition funding (JETP, ETM, etc.), "
    "     or nuclear research / SMR deployment plans in Indonesia.\n"
    "   - If any section discusses these topics in the Indonesian context, classify as TRUE and provide an exact quote.\n"
    "2. IRRELEVANT (relevancy = false):\n"
    "   - Pure foreign power plant events (e.g. Chernobyl, Fukushima) with zero relevance to Indonesian policy.\n"
    "   - Routine general crime, court hearings, or law enforcement investigations where 'PLTU' or 'batu bara' "
    "     is merely cited as a passive piece of evidence or background case without substantive discussion of energy policy.\n"
    "   - General environmental news (e.g. general beach cleanups, illegal logging, flood management) without coal or nuclear power focus.\n"
    "3. EVIDENCE: If relevant, provide an EXACT verbatim excerpt from the text in 'evidence_quote'.\n\n"
)

def evaluate_relevancy_batch(batch_df, max_retries=2):
    """
    Mengirim batch ke model dengan penanganan error adaptif:
    - Transient 500 error: Dicoba ulang 2 kali dengan jeda pendek.
    - Hard Quota error (429 / RESOURCE_EXHAUSTED): Langsung melempar QuotaExhaustedError agar loop berhenti aman.
    """
    prompt = GENERALIZED_SYSTEM_PROMPT + "Analyze the following items:\n"
    for idx, row in batch_df.iterrows():
        kwic_context = extract_topic_salient_context(row.get('title', ''), row.get('content', ''))
        prompt += f"--- Item ID: {idx} ---\nTitle: {row.get('title', '')}\nExtracted Context: {kwic_context}\n\n"

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model='gemma-4-26b-a4b-it',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=RelevancyBatchResponse,
                    temperature=0.1
                )
            )
            parsed_response = RelevancyBatchResponse.model_validate_json(response.text)
            return [
                {
                    "id": item.id,
                    "relevancy": item.relevancy,
                    "confidence": item.confidence,
                    "evidence_quote": item.evidence_quote,
                    "reasoning": item.reasoning
                }
                for item in parsed_response.results
            ]
        except Exception as e:
            err_msg = str(e).upper()
            # Deteksi apakah kuota API habis (Hard Limit)
            if "RESOURCE_EXHAUSTED" in err_msg or "QUOTA" in err_msg or "LIMIT EXCEEDED" in err_msg:
                raise QuotaExhaustedError(f"Batas kuota API tercapai: {e}")

            print(f"[Peringatan] Batch percobaan {attempt}/{max_retries} mengalami transient error: {e}")
            if attempt < max_retries:
                backoff_time = 3 * attempt
                print(f"Menunggu {backoff_time} detik sebelum mencoba ulang...")
                time.sleep(backoff_time)
            else:
                print("[Error] Percobaan batch gagal setelah retry. Melempar error untuk fallback.")
                raise e


## 5. Eksekusi Stateful & Resumable (Dapat Dilanjutkan Kapan Saja)
* **Idempotent Resume**: Jika proses berhenti (misal karena batas kuota API tercapai), saat dijalankan kembali program **otomatis melanjutkan dari baris terakhir** tanpa mengulang dari awal (0 kuota terbuang).
* **Incremental Checkpointing**: Menyimpan hasil prediksi ke disk (`relevancy_checkpoint.csv`) langsung setelah setiap batch selesai.

In [5]:
import os

CHECKPOINT_FILE = 'relevancy_checkpoint.csv'
batch_size = 5

if len(df_matched) > 0 and 'title' in df_matched.columns:
    # 1. Muat checkpoint yang sudah ada jika proses sebelumnya terhenti
    if os.path.exists(CHECKPOINT_FILE):
        try:
            df_existing = pd.read_csv(CHECKPOINT_FILE)
            already_processed_ids = set(df_existing['id'].dropna().astype(int))
            print(f"[Resume] Ditemukan file checkpoint: {len(already_processed_ids)} artikel sudah selesai dianotasi.")
        except Exception:
            already_processed_ids = set()
            df_existing = pd.DataFrame()
    else:
        already_processed_ids = set()
        df_existing = pd.DataFrame()

    # 2. Filter artikel yang BELUM pernah diproses sama sekali
    unprocessed_mask = ~df_matched.index.isin(already_processed_ids)
    df_to_process = df_matched[unprocessed_mask].copy()
    print(f"Total artikel di dataset: {len(df_matched)} | Selesai sebelumnya: {len(already_processed_ids)} | Sisa yang akan diproses: {len(df_to_process)}")

    quota_limit_reached = False
    unannotated_indices = []

    if len(df_to_process) > 0:
        # Pisahkan sisa artikel: Fast-Path vs Kandidat API
        fastpath_records = []
        candidate_indices = []

        for idx, row in df_to_process.iterrows():
            combined_text = str(row.get('title', '')) + ' ' + str(row.get('content', ''))
            if not KEYWORDS_PATTERN.search(combined_text):
                fastpath_records.append({
                    'id': idx,
                    'title': row.get('title', ''),
                    'news_relevancy': False,
                    'confidence': 1.0,
                    'evidence_quote': '',
                    'reasoning': 'Fast-path local filter: Tidak ada mention kata kunci energi/PLTU/PLTN.'
                })
            else:
                candidate_indices.append(idx)

        # Simpan hasil fast-path lokal terlebih dahulu ke checkpoint
        if len(fastpath_records) > 0:
            df_fp = pd.DataFrame(fastpath_records)
            if os.path.exists(CHECKPOINT_FILE):
                df_fp.to_csv(CHECKPOINT_FILE, mode='a', header=False, index=False)
            else:
                df_fp.to_csv(CHECKPOINT_FILE, mode='w', header=True, index=False)
            print(f"[Fast-Path] {len(df_fp)} artikel diselesaikan lokal tanpa memanggil API.")

        # Proses kandidat yang membutuhkan API dalam micro-batch
        df_candidates = df_to_process.loc[candidate_indices]
        print(f"Memulai evaluasi cloud API pada {len(df_candidates)} artikel kandidat...")

        for i in range(0, len(df_candidates), batch_size):
            batch_df = df_candidates.iloc[i:i+batch_size]
            try:
                batch_results = evaluate_relevancy_batch(batch_df)
                if len(batch_results) > 0:
                    batch_rows = []
                    for item in batch_results:
                        it_id = int(item['id'])
                        batch_rows.append({
                            'id': it_id,
                            'title': df_matched.loc[it_id, 'title'] if it_id in df_matched.index else '',
                            'news_relevancy': bool(item['relevancy']),
                            'confidence': float(item.get('confidence', 1.0)),
                            'evidence_quote': str(item.get('evidence_quote', '')),
                            'reasoning': str(item.get('reasoning', ''))
                        })
                    # Langsung append batch yang berhasil ke disk (Incremental Checkpoint)
                    df_batch_out = pd.DataFrame(batch_rows)
                    if os.path.exists(CHECKPOINT_FILE):
                        df_batch_out.to_csv(CHECKPOINT_FILE, mode='a', header=False, index=False)
                    else:
                        df_batch_out.to_csv(CHECKPOINT_FILE, mode='w', header=True, index=False)
                    print(f"Progress: {min(i + batch_size, len(df_candidates))} / {len(df_candidates)} kandidat selesai...")
                time.sleep(3)  # Pacing jeda antar batch
            except QuotaExhaustedError as qe:
                print(f"\n[PERINGATAN QUOTA] {qe}")
                print("Menghentikan eksekusi segera untuk mencegah error berulang...")
                quota_limit_reached = True
                # Catat artikel yang belum sempat dianotasi
                unannotated_indices = candidate_indices[i:]
                break
            except Exception as e_unhandled:
                print(f"\n[Error Tidak Terduga pada Batch]: {e_unhandled}")
                print("Menyimpan progress yang sudah ada dan menghentikan proses...")
                unannotated_indices = candidate_indices[i:]
                break

    # 3. Muat seluruh checkpoint final dari disk untuk evaluasi
    if os.path.exists(CHECKPOINT_FILE):
        df_final_checkpoint = pd.read_csv(CHECKPOINT_FILE).drop_duplicates(subset=['id']).reset_index(drop=True)
        # Gabungkan hasil relevansi ke df_matched
        res_map = dict(zip(df_final_checkpoint['id'], df_final_checkpoint['news_relevancy']))
        df_matched['news_relevancy'] = df_matched.index.map(res_map)

        print(f"\n=== STATUS CHECKPOINT RESUME ===")
        print(f"Total artikel selesai dianotasi : {len(df_final_checkpoint)} / {len(df_matched)}")
        if quota_limit_reached or len(unannotated_indices) > 0:
            print(f"[INFO PENTING] Tersisa {len(unannotated_indices)} artikel yang BELUM dianotasi.")
            print(f"Daftar Indeks ID yang belum selesai: {unannotated_indices[:15]}...")
            print("Saat kuota API di-reset atau model siap kembali, jalankan ulang sel ini:")
            print("Program akan otomatis melanjutkan sisa artikel tanpa memproses ulang yang sudah selesai!")
        else:
            print("[SUKSES LENGKAP] Seluruh artikel di dataset telah selesai dianotasi 100%!")
        display(df_final_checkpoint[['title', 'news_relevancy', 'confidence']].head(10))
    else:
        print('Checkpoint belum terbentuk.')
else:
    print('Tidak ada artikel yang diproses karena df_matched kosong.')


Total artikel di dataset: 42 | Selesai sebelumnya: 0 | Sisa yang akan diproses: 42
[Fast-Path] 5 artikel diselesaikan lokal tanpa memanggil API.
Memulai evaluasi cloud API pada 37 artikel kandidat...
Progress: 5 / 37 kandidat selesai...
[Peringatan] Batch percobaan 1/2 mengalami transient error: 500 INTERNAL. {'error': {'code': 500, 'message': 'Internal error encountered.', 'status': 'INTERNAL'}}
Menunggu 3 detik sebelum mencoba ulang...
Progress: 10 / 37 kandidat selesai...
Progress: 15 / 37 kandidat selesai...
Progress: 20 / 37 kandidat selesai...
Progress: 25 / 37 kandidat selesai...
Progress: 30 / 37 kandidat selesai...
Progress: 35 / 37 kandidat selesai...
Progress: 37 / 37 kandidat selesai...

=== STATUS CHECKPOINT RESUME ===
Total artikel selesai dianotasi : 42 / 42
[SUKSES LENGKAP] Seluruh artikel di dataset telah selesai dianotasi 100%!


,title,news_relevancy,confidence
0,Kapolsek Muara Batang Gadis Dicopot Usai Didug...,False,1.0
1,"Mulai Hari Ini, Jangan Biarkan Energi Anda Hab...",False,1.0
2,Kemenhut Tangkap DPO Pemodal Ilegal Logging di...,False,1.0
3,"Tenaga Ahli Menkes Temui Sultan HB X, Bicara s...",False,1.0
4,"Jemaah Haji Aceh 2026: Tertua 101 Tahun, Termu...",False,1.0
5,"PLTM Santong Hasilkan 3,5 Juta kWh Per Tahun d...",False,1.0
6,"Menteri LHK: Setelah Hujan, Kualitas Udara di ...",True,1.0
7,YLBHI Sebut Ada 26 Kasus Pencemaran Udara pada...,True,1.0
8,"Cegah Listrik Mati Bergilir, PLTU Dimodifikasi...",True,1.0
9,"Ekspor Batu Bara Dilarang, Ini Kata Adaro",True,1.0


## 6. Evaluasi Metrik & Disaggregated Error Analysis
* **Benchmark Ground Truth**: `news baru - News fixx.csv` (`Stance1`).
* **Aturan Ground Truth Biner**:
  * `Stance1` in [`SUPPORT`, `OPPOSE`, `NEUTRAL`] -> **RELEVANT** (`True`).
  * `Stance1` == `Irrelevant` (atau kosong) -> **IRRELEVANT** (`False`).
* **Evaluasi Parsial Otomatis**: Jika proses berhenti sebagian karena kuota, evaluasi tetap dapat berjalan untuk artikel yang sudah selesai di-checkpoint.

In [6]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, classification_report

if 'Stance1' in df_annot.columns and 'title' in df_annot.columns and os.path.exists(CHECKPOINT_FILE):
    df_final_checkpoint = pd.read_csv(CHECKPOINT_FILE).drop_duplicates(subset=['id'])

    # Tentukan Ground Truth Biner
    def get_ground_truth(val):
        if pd.isna(val):
            return False
        return str(val).strip().upper() in ['SUPPORT', 'OPPOSE', 'NEUTRAL']

    df_annot['ground_truth'] = df_annot['Stance1'].apply(get_ground_truth)

    # Gabungkan prediksi checkpoint dengan Ground Truth berdasarkan judul berita
    merge_cols_annot = [c for c in ['no', 'title', 'keyword', 'target', 'Stance1', 'Evidence1', 'ground_truth'] if c in df_annot.columns]

    df_eval = pd.merge(
        df_annot[merge_cols_annot],
        df_final_checkpoint[['title', 'news_relevancy', 'confidence', 'evidence_quote', 'reasoning']],
        on='title',
        how='inner'
    )

    # Buang baris yang nilai news_relevancy-nya masih NaN / belum teranotasi
    df_eval = df_eval.dropna(subset=['news_relevancy'])
    print(f'Total artikel yang dievaluasi (sudah selesai dianotasi): {len(df_eval)}')

    if len(df_eval) > 0:
        y_true = df_eval['ground_truth'].values
        y_pred = df_eval['news_relevancy'].astype(bool).values

        # Hitung Confusion Matrix
        cm = confusion_matrix(y_true, y_pred, labels=[False, True])
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
        else:
            tn = fp = fn = tp = 0

        # Metrik Klasifikasi
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1_binary = f1_score(y_true, y_pred, zero_division=0)
        f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)

        print('\n' + '='*60)
        print('                 CONFUSION MATRIX')
        print('='*60)
        print(f'                     Pred Irrelevant    Pred Relevant')
        print(f'Actual Irrelevant         {tn:<18} {fp}')
        print(f'Actual Relevant           {fn:<18} {tp}')
        print('='*60)
        print(f'Precision (Relevant)         : {precision:.4f}')
        print(f'Recall (Relevant)            : {recall:.4f}')
        print(f'F1-Score (Relevant / Binary) : {f1_binary:.4f}')
        print(f'F1-Score (Micro)             : {f1_micro:.4f}')
        print('='*60)

        # Classification Report
        print('\nClassification Report:')
        print(classification_report(y_true, y_pred, target_names=['Irrelevant', 'Relevant'], zero_division=0))

        # =========================================================================
        # DISAGGREGATED ERROR ANALYSIS (FALSE NEGATIVES & FALSE POSITIVES)
        # =========================================================================
        df_eval['error_type'] = 'Correct'
        df_eval.loc[(df_eval['ground_truth'] == True) & (df_eval['news_relevancy'] == False), 'error_type'] = 'False Negative'
        df_eval.loc[(df_eval['ground_truth'] == False) & (df_eval['news_relevancy'] == True), 'error_type'] = 'False Positive'

        errors = df_eval[df_eval['error_type'] != 'Correct']
        print(f'\n[Audit Error] Ditemukan {len(errors)} kasus mismatch ({fn} False Negatives, {fp} False Positives):')
        if len(errors) > 0:
            cols_show = [c for c in ['no', 'title', 'error_type', 'Stance1', 'Evidence1', 'evidence_quote', 'reasoning'] if c in errors.columns]
            display(errors[cols_show])
        else:
            print('Sempurna! Tidak ada False Negative maupun False Positive.')
    else:
        print('Tidak ada artikel yang cocok untuk dievaluasi.')
else:
    print('Dataset atau checkpoint belum lengkap untuk dihitung metrik evaluasinya.')


Total artikel yang dievaluasi (sudah selesai dianotasi): 42

                 CONFUSION MATRIX
                     Pred Irrelevant    Pred Relevant
Actual Irrelevant         10                 2
Actual Relevant           3                  27
Precision (Relevant)         : 0.9310
Recall (Relevant)            : 0.9000
F1-Score (Relevant / Binary) : 0.9153
F1-Score (Micro)             : 0.8810

Classification Report:
              precision    recall  f1-score   support

  Irrelevant       0.77      0.83      0.80        12
    Relevant       0.93      0.90      0.92        30

    accuracy                           0.88        42
   macro avg       0.85      0.87      0.86        42
weighted avg       0.88      0.88      0.88        42


[Audit Error] Ditemukan 5 kasus mismatch (3 False Negatives, 2 False Positives):


,no,title,error_type,Stance1,Evidence1,evidence_quote,reasoning
0,1,"PLTM Santong Hasilkan 3,5 Juta kWh Per Tahun d...",False Negative,Neutral,"saat ini, Heny melanjutkan, total pasokan dari...",NaN,The article discusses microhydro power plants ...
14,9,Pemdes Sangrawayang Tolak Pandawara Group Bers...,False Negative,Neutral,"""(Sampah berasal dari), Sungai Cimandiri karen...",NaN,The article focuses on a local dispute regardi...
18,11,Joe Biden Puji Jokowi: RI Jadi Pemain Penting ...,False Negative,Oppose,"Di gelaran KTT G20 2022 yang lalu di Bali, Joe...",NaN,The article focuses on general energy transiti...
21,14,Fakta-fakta Baru Kasus Febrie: Hotman Pengacar...,False Positive,Irrelevant,NaN,pengadaan batu bara untuk PLTU masih berstatus...,The article mentions coal procurement for PLTU...
22,15,"BRIN Kena Efisiensi Rp 1,4 T, Perpusnas Potong...",False Positive,Irrelevant,NaN,fasilitas riset seperti reaktor nuklir tetap aman,The article discusses the safety and continued...


## 7. Flowchart Metodologi Evidence-Grounded Relevancy (Mermaid Diagram)

In [7]:
import base64
from IPython.display import Image, display

def render_mermaid(graph_code):
    graph_bytes = graph_code.encode('ascii')
    base64_bytes = base64.b64encode(graph_bytes)
    base64_string = base64_bytes.decode('ascii')
    display(Image(url='https://mermaid.ink/img/' + base64_string))

render_mermaid("""
graph TD
    %% Custom Styling
    classDef startStyle fill:#2b2d42,stroke:#1d1e2c,stroke-width:2px,color:#ffffff,font-weight:bold,rx:8px,ry:8px;
    classDef filterStyle fill:#f39c12,stroke:#d68910,stroke-width:2px,color:#ffffff,font-weight:bold,rx:6px,ry:6px;
    classDef fastpathStyle fill:#457b9d,stroke:#1d3557,stroke-width:2px,color:#ffffff,font-weight:bold,rx:6px,ry:6px;
    classDef processStyle fill:#edf2f4,stroke:#8d99ae,stroke-width:2px,color:#2b2d42,rx:6px,ry:6px;
    classDef apiStyle fill:#3a86ff,stroke:#004e92,stroke-width:2px,color:#ffffff,font-weight:bold,rx:6px,ry:6px;
    classDef verifyStyle fill:#06d6a0,stroke:#0f8b6d,stroke-width:2px,color:#ffffff,font-weight:bold,rx:6px,ry:6px;
    classDef evalStyle fill:#8338ec,stroke:#5c12b7,stroke-width:2px,color:#ffffff,font-weight:bold,rx:8px,ry:8px;

    %% Nodes
    A[Muat Dataset: news baru - News fixx.csv]:::startStyle
    B{Local Lexical Scan: Ada Kata Kunci?}:::filterStyle
    C[Fast-Path Lokal: Langsung Irrelevant - 0 API Calls!]:::fastpathStyle
    D[Kompresor KWIC: Ekstrak Jendela Konteks Relevan - Hemat 73% Token]:::processStyle
    E[Micro-Batching Isi 5 dengan Pacing Sleep 4s]:::processStyle
    F[Cloud LLM Gemma-4-26B: Prediksi Relevansi, Evidence & Reasoning]:::apiStyle
    G[Verifikasi String: Validasi Evidence Quote ke Teks Asli]:::verifyStyle
    H[Gabungkan Hasil Fast-Path & Cloud ke relevancy_checkpoint.csv]:::processStyle
    I[Evaluasi: Confusion Matrix, Precision, Recall, F1 & Error Audit]:::evalStyle

    %% Flow Connections
    A --> B
    B -->|Tidak Ada Mention| C
    B -->|Ada Mention| D
    D --> E
    E --> F
    F --> G
    C --> H
    G --> H
    H --> I
""")
